In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rubin_sim.moving_objects as mo
import rubin_sim.maf as maf
import rubin_sim.phot_utils as phot_utils
from rubin_sim.data import get_data_dir, get_baseline

## Colors of SSO objects ## 

One of the first questions about the SSO objects is what colors are we using? 

For the simulations, we have a set of SEDs that can be applied to each object, which then result in a (fixed) color. In reality, the observed color is likely to vary with orbital phase (as the phase curve parameters vary by wavelength) and potentially with rotation phase -- we do not include either of these. When observing, there are also small changes in the atmospheric transmission that result in changes in the observed colors, but we are also not simulating these. We just apply a fixed color term, dependent on the SED.

In [ ]:
# Seds available:
sed_dir = os.path.join(get_data_dir(), "movingObjects")
os.listdir(sed_dir)

In [ ]:
# The kurucz_sun is included, as this is used to transform the reflectance spectra (from Binzel) to observed SEDs.
# The harris_v.dat file is included as the 'anchor' V band filter used to interpret the
#  openorb-generated "V" band magnitudes.

In [ ]:
# Read each of these SEDs, along with the LSST throughput curves, to calculate LSST colors.

ast_types = []
for s in os.listdir(sed_dir):
    if s.startswith("harris") or s.startswith("kurucz"):
        continue
    else:
        ast_types.append(s)

ast_seds = {}
for s in ast_types:
    name = s.split(".")[0]
    ast_seds[name] = phot_utils.Sed()
    ast_seds[name].read_sed_flambda(os.path.join(sed_dir, s))


lsst = {}
filterlist = ["u", "g", "r", "i", "z", "y"]
for f in filterlist:
    lsst[f] = phot_utils.Bandpass()
    lsst[f].read_throughput(os.path.join(get_data_dir(), "throughputs", "baseline", f"total_{f}.dat"))
harris = phot_utils.Bandpass()
harris.read_throughput(os.path.join(sed_dir, "harris_V.dat.gz"))

In [ ]:
# Plots can be nice.
plt.figure(figsize=(8, 5))
for f in filterlist:
    plt.plot(lsst[f].wavelen, lsst[f].sb, label=f)
# The harris filter is without atmosphere (the peak is >0.8), so rescale for visualization
plt.plot(harris.wavelen, harris.sb / 2, color="k", linestyle="--", label="Harris V")
for s in ("C", "S", "D", "TNO"):
    plt.plot(ast_seds[s].wavelen, ast_seds[s].flambda / ast_seds[s].flambda.max() / 2, linestyle=":", label=s)
plt.legend(loc=(1.01, 0.2))
plt.ylim(0, 0.5)
plt.xlim(300, 1100)
plt.grid(alpha=0.3)

In [ ]:
# Calculate the colors
# Normalize each sed so that mag in r = 0 .. because I'm not sure what colors you need and this should be easy
mags = {}
for s in ast_seds:
    fluxNorm = ast_seds[s].calc_flux_norm(0, lsst["r"])
    ast_seds[s].multiply_flux_norm(fluxNorm)
    mags[s] = {}
    for f in filterlist:
        # This calculates an AB magnitude in each filter for each SED
        mags[s][f] = ast_seds[s].calc_mag(lsst[f])
    mags[s]["V"] = ast_seds[s].calc_mag(harris)

# Turn this into a dataframe - you may find this easier to work with
d = pd.DataFrame(mags).T
d

## Calculating magnitudes ##

A second question is, how could you use some of the magnitude 'stackers' from MAF, to turn the data columns in the outputs from movingObjects into magnitudes, for comparison to other post-processing or just to access that information. 


In [ ]:
# Generate a bit of output from moving objects (or read it from disk)

orbits = mo.Orbits()
orbitFile = os.path.join(get_data_dir(), "orbits", "granvik_5k.txt")
orbits.read_orbits(orbitFile)
# Let's cut the orbits down a bit, for faster processing in the notebook here
# And we'll pick the biggest objects, to be more likely to be above the SNR limit later
orbits.orbits = orbits.orbits.query("H < 19")[0:10]

In [ ]:
orbits.orbits

In [ ]:
opsim_file = get_baseline()
print(opsim_file)
colmap = maf.get_col_map(opsim_file)
# Read just the first year of the opsim data
simdata = mo.read_observations(opsim_file, colmap, constraint="night<365")
pd.DataFrame(simdata[0:10])

In [ ]:
obsFile = "tmp_obs"
mo.run_obs(
    orbits,
    simdata,
    colmap,
    obsFile,
    footprint="camera",
    r_fov=1.75,
    eph_mode="nbody",
    prelim_eph_mode="nbody",
    obs_code="I11",
    eph_type="basic",
    t_step=1,
    rough_tol=10,
    obs_metadata="FirstYear",
)

In [ ]:
!head -15 $obsFile

In [ ]:
# Could just skip to here to read things back from disk .. now heading over to MAF

In [ ]:
# Read observations from disk.
obs = pd.read_csv(obsFile, delim_whitespace=True, comment="#")
obs[0:3].T

In [ ]:
# Add apparent magnitude with MAF stacker
magStacker = maf.MoMagStacker(
    magtype="asteroid",  # You could set this to comet_oort, for cometary oort-style
    v_mag_col="magV",
    color_col="dmag_color",
    loss_col="dmag_detect",
    m5_col="fiveSigmaDepth",
    seeing_col="seeingFwhmGeom",
    filter_col="filter",
    gamma=0.038,
    sigma=0.12,
    random_seed=58,
)

In [ ]:
# Need to set Href (value of H from orbit file) and Hval (desired value for H)
# For non-cloning, just set these to be equal
# Note that obs must be a numpy recarray at this point
Href = 0
Hval = 0
obs_stacked = magStacker.run(obs.to_records(), Href, Hval)
# Convert back to pandas DataFrame here, because it's prettier
obs_stacked = pd.DataFrame(obs_stacked).drop("index", axis=1)
obs_stacked[0:3].T

In [ ]:
# Let's skip to some observations which would be visible
# (vis = 0 or 1, 0 = probabilitistically chosen not-visible, 1 = probabilisticially chosen visible)
obs_stacked.query("vis == 1")[
    [
        "obj_id",
        "time",
        "filter",
        "fiveSigmaDepth",
        "magV",
        "dmag_color",
        "dmag_detect",
        "appMag",
        "SNR",
        "vis",
    ]
]